# QOuLiPo: Fine-tune E5-large for Thread Discrimination

**Problem:** E5-large produces uniformly high similarity (~0.90) across all OuLiPo pages, regardless of thread overlap. Pages with 0 shared threads: sim=0.87. Pages with 5 shared threads: sim=0.93. Gap too small.

**Solution:** Contrastive fine-tuning. Teach the model that pages sharing ≥4 threads should be similar (label=1.0) and pages sharing ≤2 threads should be dissimilar (label=0.0).

**Data:** ~400 pages from 10 OuLiPo constrained texts, each with 5-6 thread labels.

**Runtime:** ~20 min on Colab T4 GPU.

In [ ]:
!pip install -q sentence-transformers datasets accelerate pulp

In [ ]:
# Mount Google Drive to access texts
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = 'generation'   # the qoulipo/generation folder (mount or copy it in Colab)
TEXTS_DIR = os.path.join(BASE, 'texts')
THREADS_DIR = os.path.join(BASE, 'threads')
OUTPUT_MODEL = os.path.join(BASE, 'models', 'e5-large-oulipo')

print(f'Texts: {len([f for f in os.listdir(TEXTS_DIR) if f.endswith(".md")])} md files')

In [ ]:
import re, json, random
from pathlib import Path
from collections import defaultdict
import numpy as np

def parse_pages(text_path):
    text = Path(text_path).read_text(encoding='utf-8')
    pages = []
    parts = re.split(r'####\s+PAGE\s+(\d+)', text)
    for i in range(1, len(parts), 2):
        page_num = int(parts[i])
        body = parts[i + 1]
        m = re.search(r'\*Threads?:\s*(.+?)\*', body)
        threads = [t.strip() for t in m.group(1).split(',')] if m else []
        lines = body.strip().split('\n')
        text_lines = [l for l in lines if not l.startswith('*') and l.strip()]
        page_text = re.sub(r'\n---\s*$', '', '\n'.join(text_lines).strip()).strip()
        if page_text and threads:
            pages.append({'page': page_num, 'threads': set(threads),
                         'text': page_text, 'source': Path(text_path).stem})
    return pages

# Load all texts
skip = {'parity_sweep.md', 'topic_clusters.md'}
all_pages = []
for f in sorted(Path(TEXTS_DIR).glob('*.md')):
    if f.name in skip or f.name.startswith('oulipo'):
        continue
    pages = parse_pages(f)
    if pages:
        all_pages.extend(pages)
        print(f'  {f.name}: {len(pages)} pages')

print(f'\nTotal: {len(all_pages)} pages from {len(set(p["source"] for p in all_pages))} texts')

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

# Split train/eval
rng = random.Random(42)
indices = list(range(len(all_pages)))
rng.shuffle(indices)
split = int(0.85 * len(all_pages))
train_pages = [all_pages[i] for i in indices[:split]]
eval_pages = [all_pages[i] for i in indices[split:]]
print(f'Train: {len(train_pages)}, Eval: {len(eval_pages)}')

# Build contrastive pairs
examples = []
for pos_ov, neg_ov in [(4, 2), (3, 1), (5, 2)]:
    for i in range(len(train_pages)):
        for j in range(i+1, len(train_pages)):
            ov = len(train_pages[i]['threads'] & train_pages[j]['threads'])
            if ov >= pos_ov:
                examples.append(InputExample(
                    texts=['passage: ' + train_pages[i]['text'],
                           'passage: ' + train_pages[j]['text']], label=1.0))
            elif ov <= neg_ov:
                examples.append(InputExample(
                    texts=['passage: ' + train_pages[i]['text'],
                           'passage: ' + train_pages[j]['text']], label=0.0))

pos = [e for e in examples if e.label > 0.5]
neg = [e for e in examples if e.label <= 0.5]
neg = neg[:len(pos) * 2]  # 1:2 ratio
balanced = pos + neg
rng.shuffle(balanced)
print(f'Pairs: {len(pos)} pos + {len(neg)} neg = {len(balanced)} total')

# Eval set
es1, es2, esc = [], [], []
for i in range(len(eval_pages)):
    for j in range(i+1, len(eval_pages)):
        ov = len(eval_pages[i]['threads'] & eval_pages[j]['threads'])
        if ov >= 4 or ov <= 2:
            es1.append('passage: ' + eval_pages[i]['text'])
            es2.append('passage: ' + eval_pages[j]['text'])
            esc.append(1.0 if ov >= 4 else 0.0)
evaluator = EmbeddingSimilarityEvaluator(es1[:1000], es2[:1000], esc[:1000], name='thread-overlap')

In [ ]:
# Train
model = SentenceTransformer('intfloat/multilingual-e5-large')

train_dl = DataLoader(balanced, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

os.makedirs(OUTPUT_MODEL, exist_ok=True)

model.fit(
    train_objectives=[(train_dl, train_loss)],
    evaluator=evaluator,
    epochs=5,
    warmup_steps=int(0.1 * len(train_dl)),
    output_path=OUTPUT_MODEL,
    show_progress_bar=True,
    evaluation_steps=len(train_dl),
)
print(f'\n✓ Model saved to {OUTPUT_MODEL}')

In [ ]:
# Validate on rho=1.0 text
from sklearn.metrics.pairwise import cosine_similarity

rho1 = [p for p in all_pages if p['source'] == 'livre_irremplacable']
N = len(rho1)
print(f'Validating livre_irremplacable: {N} pages')

embs = model.encode(['passage: ' + p['text'] for p in rho1],
                     batch_size=32, normalize_embeddings=True)
sim = cosine_similarity(embs)
np.fill_diagonal(sim, 0)
upper = sim[np.triu_indices(N, k=1)]

print(f'Sim: mean={np.mean(upper):.4f}, std={np.std(upper):.4f}')
print(f'Min={np.min(upper):.4f}, Max={np.max(upper):.4f}')
print(f'% > 0.78: {100*np.mean(upper > 0.78):.1f}%')

# Thread overlap vs similarity
by_ov = defaultdict(list)
for i in range(N):
    for j in range(i+1, N):
        ov = len(rho1[i]['threads'] & rho1[j]['threads'])
        by_ov[ov].append(sim[i][j])

print('\nOverlap vs similarity (FINE-TUNED):')
for ov in sorted(by_ov):
    s = by_ov[ov]
    pct = 100 * np.mean(np.array(s) > 0.78)
    print(f'  Overlap {ov}: n={len(s):4d}, sim={np.mean(s):.4f} ± {np.std(s):.4f}, >0.78: {pct:.1f}%')

In [ ]:
# Graph + MIS analysis at different thresholds
import networkx as nx
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, PULP_CBC_CMD

node_ids = [f'p{p["page"]}' for p in rho1]

print('Threshold sweep:')
for thresh in [0.40, 0.50, 0.60, 0.70, 0.78, 0.85]:
    G = nx.Graph()
    G.add_nodes_from(node_ids)
    for i in range(N):
        sims = sim[i].copy(); sims[i] = -1
        top_k = np.argsort(sims)[-8:]
        for j in top_k:
            if sims[j] >= thresh:
                G.add_edge(node_ids[i], node_ids[j])
    
    prob = LpProblem('MIS', LpMaximize)
    x = {n: LpVariable(f'x_{n}', cat='Binary') for n in G.nodes()}
    prob += lpSum(x.values())
    for u, v in G.edges(): prob += x[u] + x[v] <= 1
    prob.solve(PULP_CBC_CMD(msg=0))
    mis = [n for n, v in x.items() if v.varValue > 0.5]
    
    # Quick enumerate
    sols = []
    for it in range(30):
        prob = LpProblem(f'E_{it}', LpMaximize)
        x = {n: LpVariable(f'x_{n}', cat='Binary') for n in G.nodes()}
        prob += lpSum(x.values())
        for u, v in G.edges(): prob += x[u] + x[v] <= 1
        for prev in sols: prob += lpSum(x[n] for n in prev) <= len(prev) - 1
        prob.solve(PULP_CBC_CMD(msg=0))
        sol = sorted([n for n, v in x.items() if v.varValue > 0.5])
        if not sol or (sols and len(sol) < len(sols[0])): break
        sols.append(sol)
    
    ess = set.intersection(*[set(s) for s in sols]) if sols else set()
    rho = len(ess) / len(sols[0]) if sols else 0
    print(f'  thresh={thresh}: E={G.number_of_edges()}, MIS={len(mis)}, sols={len(sols)}, rho={rho:.3f}')

In [ ]:
# Compare BEFORE vs AFTER fine-tuning
model_orig = SentenceTransformer('intfloat/multilingual-e5-large')
embs_orig = model_orig.encode(['passage: ' + p['text'] for p in rho1],
                               batch_size=32, normalize_embeddings=True)
sim_orig = cosine_similarity(embs_orig)
np.fill_diagonal(sim_orig, 0)

print('BEFORE fine-tuning:')
for ov in sorted(by_ov):
    vals = [sim_orig[i][j] for i in range(N) for j in range(i+1, N)
            if len(rho1[i]['threads'] & rho1[j]['threads']) == ov]
    print(f'  Overlap {ov}: sim={np.mean(vals):.4f} ± {np.std(vals):.4f}')

print('\nAFTER fine-tuning:')
for ov in sorted(by_ov):
    s = by_ov[ov]
    print(f'  Overlap {ov}: sim={np.mean(s):.4f} ± {np.std(s):.4f}')

# Gap improvement
gap_before = np.mean([sim_orig[i][j] for i in range(N) for j in range(i+1,N)
                      if len(rho1[i]['threads']&rho1[j]['threads'])>=4]) - \
             np.mean([sim_orig[i][j] for i in range(N) for j in range(i+1,N)
                      if len(rho1[i]['threads']&rho1[j]['threads'])<=2])
gap_after = np.mean(by_ov.get(4,[])+by_ov.get(5,[])) - \
            np.mean(by_ov.get(0,[])+by_ov.get(1,[])+by_ov.get(2,[]))
print(f'\nGap (overlap≥4 vs ≤2): before={gap_before:.4f}, after={gap_after:.4f}')
print(f'Improvement: {gap_after/gap_before:.1f}x')

## Results

If the fine-tuning works:
- The gap between overlap ≥4 and overlap ≤2 should increase from ~0.06 to >0.15
- The % of pairs above 0.78 should drop below 100%
- The rho at k=8 should increase from ~0.15 toward 1.0

The fine-tuned model is saved to Google Drive at `models/e5-large-oulipo/` and can be loaded with:
```python
model = SentenceTransformer('path/to/models/e5-large-oulipo')
```